<a href="https://colab.research.google.com/github/max-power-126/lab1_FFNN/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_circles
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
from csv import reader
import time


# D= length avg 3,E=max length 4,F=bytes 5,J=inter arrival time 9,K=packet length 10


data = pd.read_csv("dataset_lab_1.csv",encoding="utf-8")
# Modo corretto ed efficiente in pandas per filtrare i valori uguali a 0 nelle colonne desiderate
rawdata_size= data.shape
######################################## Missing and Duplicate #############################

print(f"dati iniziali",data.shape)
#drop NaN

data.dropna(inplace=True)

print(f"dati No NaN",data.shape)

# Definiamo le colonne che non devono contenere zeri
columns_to_filter = [
    'Bwd Packet Length Mean',
    'Bwd Packet Length Max',
    'Flow Bytes/s',
    'Flow IAT Mean',
    'Packet Length Mean'
]

# Creiamo una maschera booleana: per ogni riga, tutte le colonne specificate devono essere diverse da 0
# (data[col] != 0).all(axis=1) verifica che la condizione sia True per TUTTE le colonne selezionate


mask = (data[columns_to_filter] != 0.0).all(axis=1)


# Applichiamo la maschera per ottenere il dataset pulito
data_cleaned = data[mask].copy()

modified_size=data_cleaned.shape

print(f"data filtered= {modified_size}")

data_cleaned.drop_duplicates(inplace=True,keep="first")

print(f"data no duplicate= {data_cleaned.shape}")

# Mostriamo la differenza di righe prima e dopo la pulizia

# 1. Rimuoviamo i valori NaN dal dataset finale


# 2. Rimuoviamo i valori infiniti (inf e -inf)
# Selezioniamo solo le colonne numeriche per verificare la presenza di infiniti
numeric_cols = data_cleaned.select_dtypes(include=[np.number]).columns
is_infinite = np.isinf(data_cleaned[numeric_cols]).any(axis=1)

# Teniamo solo le righe che NON contengono infiniti
data_cleaned_final = data_cleaned[~is_infinite]

print(f"data no inf= {data_cleaned_final.shape}")

data_choosed=data_cleaned_final.copy()


########################################### Split #############################

## train = 60 %, validation = 20 % , test=20%

# Split the dataset
X = data_choosed[['Flow Duration', 'Flow IAT Mean', 'Fwd PSH Flags',
       'Bwd Packet Length Mean', 'Bwd Packet Length Max', 'Flow Bytes/s',
       'Down/Up Ratio', 'SYN Flag Count', 'Fwd Packet Length Mean',
       'Fwd IAT Std', 'Packet Length Mean', 'Fwd Packet Length Max',
       'Subflow Fwd Packets', 'Flow Packets/s', 'Total Fwd Packets',
       'Destination Port']].values
y = data_choosed['Label'].values

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)


#################### Scaling ################################################
# Standardize the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(y_train)
# Convert data to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

########################################################################


dati iniziali (31507, 17)
dati No NaN (31487, 17)
data filtered= (23773, 17)
data no duplicate= (23280, 17)
data no inf= (23280, 17)
['DoS Hulk' 'PortScan' 'Benign' ... 'Benign' 'Benign' 'DoS Hulk']


TypeError: can't convert np.ndarray of type numpy.object_. The only supported types are: float64, float32, float16, complex64, complex128, int64, int32, int16, int8, uint64, uint32, uint16, uint8, and bool.

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
plt.scatter(x=data["Feature_1"].values,
            y=data["Feature_2"].values,
            c=data["Label"].values,
            cmap=plt.cm.RdYlBu)
plt.show()
plt.close()

fig, ax = plt.subplots(figsize=(10,5))
plt.scatter(x=data["Feature_1"].values,
            y=data["Feature_2"].values,
            c=data["Label"].values,
            cmap=plt.cm.RdYlBu)
plt.xlim(-2,2)
plt.ylim(-2,2)
plt.show()
plt.close()

Task 2

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"The device is set to: {device}")

In [ ]:
# Convert data to PyTorch tensors and send to CUDA
X_train_tensor = torch.tensor(X_train, dtype=torch.float32).to(device)
y_train_tensor = torch.tensor(y_train, dtype=torch.long).to(device)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_tensor = torch.tensor(y_val, dtype=torch.long).to(device)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32).to(device)
y_test_tensor = torch.tensor(y_test, dtype=torch.long).to(device)

In [ ]:
# Create DataLoader
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# Define the neural network
class ffnn(nn.Module):
    def __init__(self, input_size, output_size):
        super(ffnn, self).__init__()
        self.fc1 = nn.Linear(input_size, 64)
        self.fc2 = nn.Linear(64, output_size)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

In [ ]:
# Initialize the model, loss function, and optimizer
model = ffnn(2, 3)
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=5e-4)

In [ ]:
# Inspecting the learned weights and biases
print(f'Learned weights: {model.fc1.weight.data}')
print(f'Learned biases: {model.fc1.bias.data}')